# Graph Analytics Notebook
Visualizes community detection results, PageRank, and query benchmarks from Neo4j GDS.

In [ ]:
import os
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from neo4j import GraphDatabase
from pyvis.network import Network

sns.set_theme(style='whitegrid')

_cwd = Path.cwd().resolve()
if os.environ.get("RESULTS_DIR"):
    RESULTS_DIR = Path(os.environ["RESULTS_DIR"]).expanduser().resolve()
elif (_cwd / "data").is_dir():
    RESULTS_DIR = (_cwd / "data" / "results").resolve()
else:
    RESULTS_DIR = (_cwd.parent / "data" / "results").resolve()
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

NEO4J_URI  = os.getenv('NEO4J_URI',  'bolt://localhost:7687')
NEO4J_USER = os.getenv('NEO4J_USER', 'neo4j')
NEO4J_PASS = os.getenv('NEO4J_PASSWORD', 'neo4jpassword')

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASS))

def run(q, **p):
    with driver.session() as s:
        return pd.DataFrame([dict(r) for r in s.run(q, **p)])

## 1. Community size distribution (Louvain)

In [ ]:
comm_df = run("""
    MATCH (d:Diagnosis) WHERE d.community_id IS NOT NULL
    RETURN d.community_id AS community, count(*) AS size
    ORDER BY size DESC
""")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Bar: top 20 communities
top20 = comm_df.head(20)
axes[0].bar(range(len(top20)), top20['size'], color=sns.color_palette('tab20', len(top20)))
axes[0].set_xlabel('Community (ranked by size)')
axes[0].set_ylabel('Number of diagnoses')
axes[0].set_title('Top 20 Communities by Size', fontsize=13)

# Histogram: community size distribution
axes[1].hist(comm_df['size'], bins=30, edgecolor='white')
axes[1].set_xlabel('Community size')
axes[1].set_ylabel('Count')
axes[1].set_title('Community Size Distribution', fontsize=13)

plt.suptitle(f'Louvain Communities ({len(comm_df)} total)', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'communities.png'), dpi=150, bbox_inches='tight')
plt.show()

## 2. Top communities — what diseases cluster together?

In [ ]:
top_comm_df = run("""
    MATCH (d:Diagnosis) WHERE d.community_id IS NOT NULL
    WITH d.community_id AS cid, collect(d.short_title)[..6] AS titles, count(*) AS sz
    ORDER BY sz DESC LIMIT 10
    RETURN cid, sz AS size, titles
""")

for _, row in top_comm_df.iterrows():
    print(f"Community {row['cid']:4d} ({row['size']:3d} dx): {', '.join(row['titles'])}")

## 3. PageRank — most central medications

In [ ]:
pr_df = run("""
    MATCH (m:Medication) WHERE m.pagerank IS NOT NULL
    RETURN m.drug AS drug, m.pagerank AS rank
    ORDER BY rank DESC LIMIT 25
""")

fig, ax = plt.subplots(figsize=(10, 6))
ax.barh(pr_df['drug'][::-1], pr_df['rank'][::-1], color=sns.color_palette('Blues_d', len(pr_df)))
ax.set_xlabel('PageRank score')
ax.set_title('Top 25 Medications by PageRank\n(most central in prescription network)', fontsize=13)
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'pagerank.png'), dpi=150, bbox_inches='tight')
plt.show()

## 4. Query Benchmark — latency p50/p95/p99

In [ ]:
bench = pd.read_csv(str(RESULTS_DIR / 'query_benchmark.csv'))

fig, ax = plt.subplots(figsize=(11, 5))
x = range(len(bench))
width = 0.25

ax.bar([i - width for i in x], bench['p50_ms'], width, label='p50')
ax.bar(x, bench['p95_ms'], width, label='p95')
ax.bar([i + width for i in x], bench['p99_ms'], width, label='p99')

ax.set_xticks(list(x))
ax.set_xticklabels(bench['query'], rotation=20, ha='right')
ax.set_ylabel('Latency (ms)')
ax.set_title('Query Latency Percentiles (p50/p95/p99)', fontsize=13)
ax.legend()
plt.tight_layout()
plt.savefig(str(RESULTS_DIR / 'query_latency.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. Interactive subgraph visualization (pyvis)

In [ ]:
# Sample patient subgraph
SAMPLE_PATIENT = '10006'  # Change to a real subject_id from your data

edges_df = run("""
    MATCH (p:Patient {subject_id: $sid})-[:HAS_ADMISSION]->(a:Admission)
    MATCH (a)-[:HAS_DIAGNOSIS]->(d:Diagnosis)
    OPTIONAL MATCH (a)-[:PRESCRIBED]->(m:Medication)
    RETURN 
        'Patient_' + p.subject_id AS src, 'Admission_' + a.hadm_id AS adm,
        'Diagnosis_' + d.icd9_code + ': ' + d.short_title AS diag,
        CASE WHEN m IS NOT NULL THEN 'Med: ' + m.drug ELSE null END AS med
    LIMIT 80
""", sid=SAMPLE_PATIENT)

net = Network(height='500px', width='100%', notebook=True, cdn_resources='in_line')
net.add_node('Patient_' + SAMPLE_PATIENT, color='#e74c3c', size=30, label=f'Patient\n{SAMPLE_PATIENT}')

for _, row in edges_df.iterrows():
    if row['adm']:
        net.add_node(row['adm'], color='#3498db', size=20, label=row['adm'])
        net.add_edge(row['src'], row['adm'])
    if row['diag']:
        net.add_node(row['diag'], color='#2ecc71', size=15, label=row['diag'][-30:])
        net.add_edge(row['adm'], row['diag'])
    if row['med']:
        net.add_node(row['med'], color='#f39c12', size=12, label=row['med'][-20:])
        net.add_edge(row['adm'], row['med'])

net.show('patient_subgraph.html')
print('Saved patient_subgraph.html — open in browser')

In [ ]:
driver.close()